# Landfill Waste Sorting with Deep Learning

**IT8414 Deep Learning for Data Analytics · Group 3 · Bahrain Polytechnic, Semester 1, 2026–2027**

Members: Sayed Fadhul Hashem, Ali Haji, Mohamed Alradhi  
Tutor: Dr. Shomona Jacob

This notebook classifies photos of landfill waste into nine material classes using the RealWaste dataset (see `docs/DATA.md`). It compares a custom CNN trained from scratch with MobileNetV2 transfer learning.

| Role | Lead | Contributors | Sections |
|---|---|---|---|
| A: Data preparation and evaluation | Sayed Fadhul Hashem | Ali Haji, Mohamed Alradhi | 2.0, 3.0, 4.0, 8.0 |
| B: Baseline model, training and Grad-CAM | Mohamed Alradhi | Sayed Fadhul Hashem, Ali Haji | 1.0, 5.0, 7.0, 9.1, 9.3 |
| C: Transfer learning and error analysis | Ali Haji | Sayed Fadhul Hashem, Mohamed Alradhi | 6.0, 9.2, 10.0 |

Each section opens with a short note on what the code does and why, and every code cell names the member who wrote it.

## 1.0 Setup

*Role B. Lead: Mohamed Alradhi. Contributors: Sayed Fadhul Hashem, Ali Haji.*

Imports, library versions, random seed, device and the quick re-run switch.

In [ ]:
# Author: Sayed Fadhul Hashem
# Dataset paths and class names used by every later section
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules     # True when running on Google Colab
# Colab: /content/data; laptop: the repo's git-ignored data/ folder
DATA_DIR = Path("data") if IN_COLAB else Path("../data")
ZIP_PATH = DATA_DIR / "realwaste.zip"
IMAGE_DIR = DATA_DIR / "realwaste-main" / "RealWaste"   # one folder of photos per class

DATA_URL = "https://archive.ics.uci.edu/static/public/908/realwaste.zip"
# SHA-256 fingerprint of our 30 Oct 2026 download (see docs/DATA.md)
ZIP_SHA256 = "1ede08b32358ee62065bcc1c8cb47a2ece04e8dff5b1fb53342bb8750895b2f3"

# Alphabetical, the same order as the folders: a class's position is its number label (0-8)
CLASS_NAMES = ["Cardboard", "Food Organics", "Glass", "Metal", "Miscellaneous Trash",
               "Paper", "Plastic", "Textile Trash", "Vegetation"]

## 2.0 Data loading

*Role A. Lead: Sayed Fadhul Hashem. Contributors: Ali Haji, Mohamed Alradhi.*

This section puts the RealWaste images on the machine and turns the class folders into a table with one row per image: file path, class name and number label. The download runs only once and is checked against the fingerprint in `docs/DATA.md`, so every run, including the offline demo, uses exactly the same files. We recount the images per class before any analysis, because a wrong count would change every result that follows.

In [ ]:
# Author: Sayed Fadhul Hashem
# Download and unzip RealWaste only if it is missing, so later runs and the demo work offline
import hashlib
import shutil
import urllib.request
import zipfile

DATA_DIR.mkdir(parents=True, exist_ok=True)
if not ZIP_PATH.exists():
    print("Downloading RealWaste (656.6 MB)...")
    # Browser-style header: some servers refuse Python's default user agent
    req = urllib.request.Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response, open(ZIP_PATH, "wb") as out:
        shutil.copyfileobj(response, out)   # stream to disk instead of holding 656 MB in memory

# Fingerprint check: proves we use exactly the file documented in docs/DATA.md
with open(ZIP_PATH, "rb") as f:
    digest = hashlib.file_digest(f, "sha256").hexdigest()
assert digest == ZIP_SHA256, "Zip differs from the documented download: delete it and run again"

if not IMAGE_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_DIR)
print("Fingerprint OK. Images ready in:", IMAGE_DIR.resolve())

In [ ]:
# Author: Sayed Fadhul Hashem
# Build the image table: one row per photo with its path, class name and number label
import pandas as pd

folders = sorted(p.name for p in IMAGE_DIR.iterdir() if p.is_dir())
# Stop early if the folders are not exactly our 9 classes
assert folders == CLASS_NAMES, f"Unexpected class folders: {folders}"

rows = [{"path": str(img), "class_name": name, "label": label}
        for label, name in enumerate(CLASS_NAMES)
        for img in sorted((IMAGE_DIR / name).glob("*.jpg"))]
df = pd.DataFrame(rows)
print(f"{len(df)} images in {df['class_name'].nunique()} classes")
df.head()

In [ ]:
# Author: Sayed Fadhul Hashem
# Recount each class and compare with the counts published by the dataset authors
PUBLISHED = {"Cardboard": 461, "Food Organics": 411, "Glass": 420, "Metal": 790,
             "Miscellaneous Trash": 495, "Paper": 500, "Plastic": 921,
             "Textile Trash": 318, "Vegetation": 436}
counts = pd.DataFrame({"images": df["class_name"].value_counts(),
                       "published": pd.Series(PUBLISHED)}).loc[CLASS_NAMES]
counts["match"] = counts["images"] == counts["published"]
print("Total images:", counts["images"].sum(), "| every class matches:", counts["match"].all())
counts

## 3.0 Exploratory data analysis

*Role A. Lead: Sayed Fadhul Hashem. Contributors: Ali Haji, Mohamed Alradhi.*

This section looks at the data before any modelling, and each figure leads to a design decision later in the notebook. The figures only describe the dataset. Nothing computed here is fed to the models (pixel scaling is a fixed rule, not learned from the data), so no information from the future test set leaks into training.

In [ ]:
# Author: Sayed Fadhul Hashem
# Figure 1: images per class, to see how unbalanced the classes are
import matplotlib.pyplot as plt

counts_sorted = df["class_name"].value_counts().sort_values()   # smallest at the bottom
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(counts_sorted.index, counts_sorted.values, color="tab:blue")
labels = [f"{n} ({n / len(df):.1%})" for n in counts_sorted]    # count and share of all photos
ax.bar_label(bars, labels=labels, padding=3, fontsize=8)
ax.set_xlabel("Number of images")
ax.set_title(f"Figure 1: Images per class ({len(df):,} photos)")
ax.set_xlim(0, counts_sorted.max() * 1.25)                      # room for the labels
plt.tight_layout()
plt.show()

# Two numbers that drive later decisions
ratio = counts_sorted.max() / counts_sorted.min()
majority_acc = counts_sorted.max() / len(df)
print(f"Largest / smallest class: {ratio:.1f} to 1")
print(f"Always guessing '{counts_sorted.idxmax()}' would score {majority_acc:.1%} accuracy")

**What Figure 1 shows and what we decide from it.** Plastic has 921 photos and Textile Trash only 318, a gap of 2.9 to 1, and always guessing Plastic would already score 19.4% accuracy. This leads to three decisions:
- Section 4.0 splits the data stratified by class, so training, validation and test keep these same shares.
- We report macro-averaged F1 as the main metric, because plain accuracy would hide weak small classes.
- Section 7.0 tests class weights against no weights.

In [ ]:
# Author: Sayed Fadhul Hashem
# Figure 2: privacy check - can a paper or cardboard photo show a readable name or address?
from PIL import Image

check = (df[df["class_name"].isin(["Paper", "Cardboard"])]
         .groupby("class_name").sample(n=6, random_state=42))   # the same 12 photos every run
fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
for ax, (_, row) in zip(axes.flat, check.iterrows()):
    ax.imshow(Image.open(row["path"]))
    ax.set_title(Path(row["path"]).name, fontsize=8)
    ax.axis("off")
fig.suptitle("Figure 2: Privacy check on 6 Cardboard (top) and 6 Paper (bottom) photos")
plt.tight_layout()
plt.show()

**What Figure 2 shows.** These 12 photos show packaging with brand names, crumpled paper, a rolled newspaper, a road-safety leaflet, an advertising flyer and a window envelope with a postmark. None shows a readable personal name or address, and at 524 × 524 pixels small print such as body text cannot be read. Envelopes and letters do appear in the Paper class, so the privacy risk is low but not zero; the ethics discussion (report §6.5) covers it.

## 4.0 Preprocessing and splitting

*Role A. Lead: Sayed Fadhul Hashem. Contributors: Ali Haji, Mohamed Alradhi.*

Cleaning, resizing, augmentation, and the stratified train/validation/test split with leakage checks.

## 5.0 Baseline model

*Role B. Lead: Mohamed Alradhi. Contributors: Sayed Fadhul Hashem, Ali Haji.*

A custom CNN trained from scratch.

### 5.1 Architecture

### 5.2 Training

### 5.3 Curves

## 6.0 Improved model

*Role C. Lead: Ali Haji. Contributors: Sayed Fadhul Hashem, Mohamed Alradhi.*

MobileNetV2 pretrained on ImageNet: first frozen with a new head, then fine-tuned.

### 6.1 Architecture

### 6.2 Training

### 6.3 Curves

## 7.0 Hyperparameter experiments

*Role B. Lead: Mohamed Alradhi. Contributors: Sayed Fadhul Hashem, Ali Haji.*

Each member runs their own sweeps: Sayed Fadhul Hashem (augmentation strength, class weights), Mohamed Alradhi (learning rate, batch size), Ali Haji (fine-tune depth, dropout). Every experiment is judged on the validation set only.

## 8.0 Test-set evaluation

*Role A. Lead: Sayed Fadhul Hashem. Contributors: Ali Haji, Mohamed Alradhi.*

Metrics, confusion matrix and per-class report. The test set is used once, here.

## 9.0 Interpretability and error analysis

*Roles B and C. Mohamed Alradhi leads 9.1 and 9.3; Ali Haji leads 9.2. The other members contribute to each part.*

### 9.1 Grad-CAM

*Role B. Lead: Mohamed Alradhi. Contributors: Sayed Fadhul Hashem, Ali Haji.*

### 9.2 Error analysis

*Role C. Lead: Ali Haji. Contributors: Sayed Fadhul Hashem, Mohamed Alradhi.*

### 9.3 Prediction on any image file

*Role B. Lead: Mohamed Alradhi. Contributors: Sayed Fadhul Hashem, Ali Haji.*

## 10.0 Conclusion

*Role C. Lead: Ali Haji. Contributors: Sayed Fadhul Hashem, Mohamed Alradhi.*

Summary of findings.